# UPI Fraud Detector
Run the cells in order (Colab: Runtime > Run all).

> Data is **synthetic**: fraud labels follow hand-written risk rules. Real UPI data is private.

## 1. Create synthetic data and explore it

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 50000

df = pd.DataFrame({
    "amount": np.round(rng.lognormal(mean=6.0, sigma=1.0, size=n), 2),
    "hour": rng.integers(0, 24, n),
    "txn_type": rng.choice(["P2P", "P2M", "BILL", "COLLECT_REQUEST"], n, p=[0.40, 0.40, 0.15, 0.05]),
    "is_new_receiver": rng.choice([0, 1], n, p=[0.80, 0.20]),
    "device_changed": rng.choice([0, 1], n, p=[0.95, 0.05]),
    "txns_last_1h": rng.poisson(0.5, n),
    "account_age_days": rng.integers(1, 2000, n),
    "failed_pin_attempts": rng.choice([0, 1, 2, 3], n, p=[0.93, 0.04, 0.02, 0.01]),
})

# Fraud rules (synthetic): risky patterns increase fraud probability
risk = (-7.5
        + 2.0 * (df["amount"] > 5000)
        + 1.8 * (df["hour"] <= 4)
        + 2.2 * df["is_new_receiver"]
        + 3.0 * df["device_changed"]
        + 0.9 * df["txns_last_1h"]
        + 1.8 * (df["account_age_days"] < 30)
        + 1.2 * df["failed_pin_attempts"]
        + 2.2 * (df["txn_type"] == "COLLECT_REQUEST"))
prob = 1 / (1 + np.exp(-risk))
df["is_fraud"] = (rng.random(n) < prob).astype(int)

print("Total transactions:", len(df))
print("Fraud transactions:", df["is_fraud"].sum(), f"({df['is_fraud'].mean()*100:.2f}%)")
print("\nFraud rate by transaction type (%):")
print((df.groupby("txn_type")["is_fraud"].mean() * 100).round(2))
print("\nFraud rate by device_changed (%):")
print((df.groupby("device_changed")["is_fraud"].mean() * 100).round(2))
df.head()

## 2. Train models and evaluate
Accuracy is misleading when fraud is rare. Look at precision, recall and PR-AUC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

X = pd.get_dummies(df.drop(columns="is_fraud"), columns=["txn_type"])
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced_subsample",
        min_samples_leaf=3, n_jobs=-1, random_state=42),
}

print("Accuracy if we ALWAYS say 'not fraud':", round(1 - y_test.mean(), 4), "\n")

for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    print("=====", name, "=====")
    print(classification_report(y_test, pred, digits=3))
    print("Confusion matrix [[TN FP] [FN TP]]:\n", confusion_matrix(y_test, pred))
    print("PR-AUC:", round(average_precision_score(y_test, p), 3), "\n")

rf = models["Random Forest"]
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
importance.plot(kind="barh", title="Which features detect fraud?", figsize=(7, 4))

## 3. Tune the threshold and check one transaction

In [ ]:
from sklearn.metrics import precision_recall_curve

# 1) Find the best threshold (instead of the default 0.5)
p = rf.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, p)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best = f1[:-1].argmax()
THRESHOLD = thr[best]
print(f"Best threshold: {THRESHOLD:.2f} | precision {prec[best]:.2f} | recall {rec[best]:.2f}")

pred = (p >= THRESHOLD).astype(int)
print("Confusion matrix [[TN FP] [FN TP]]:\n", confusion_matrix(y_test, pred))

# 2) Check ONE new transaction
def check_transaction(amount, hour, txn_type, is_new_receiver, device_changed,
                      txns_last_1h, account_age_days, failed_pin_attempts):
    row = pd.DataFrame([{
        "amount": amount, "hour": hour, "txn_type": txn_type,
        "is_new_receiver": is_new_receiver, "device_changed": device_changed,
        "txns_last_1h": txns_last_1h, "account_age_days": account_age_days,
        "failed_pin_attempts": failed_pin_attempts}])
    row = pd.get_dummies(row, columns=["txn_type"]).reindex(columns=X.columns, fill_value=0)
    risk_score = rf.predict_proba(row)[0, 1]
    verdict = "🚨 SUSPICIOUS - block / ask OTP" if risk_score >= THRESHOLD else "✅ Looks normal"
    print(f"Risk score: {risk_score:.2f} -> {verdict}")

# Normal payment: Rs 250 to a known shop, afternoon
check_transaction(250, 14, "P2M", 0, 0, 0, 800, 0)

# Suspicious: Rs 25,000 collect request, 3 AM, new receiver, new device, new account
check_transaction(25000, 3, "COLLECT_REQUEST", 1, 1, 4, 10, 2)